# Task 2 of 3 (`Process_Medallion`)
### Lakeflow Jobs Orchestration Lab · CDC + Medallion Architecture
**Databricks Free Edition (serverless)**

This is the **second task** in the Job. It is the core of the lab, where we apply the
**Medallion architecture to a CDC feed**.

Here we use a **Notebook task** instead of a Pipeline task for two specific Free Edition limitations:

1. **`AUTO CDC INTO`** (the declarative command that replaced `APPLY CHANGES INTO`)
   **can only run inside a** Lakeflow Declarative Pipelines **pipeline**, not within a notebook cell.
2. Free Edition allows **only one active pipeline per pipeline type**, making it fragile to
   depend on a pipeline inside the Job for a lab environment.

For this reason, we manually reproduce the same logic performed by
`AUTO CDC INTO` using **`MERGE INTO`**. At the end, we also show the equivalent official syntax.

**Depends on:** `Land_New_Data` (Task 1).

In [0]:
# Shared configuration (identical across the 3 Job tasks)
catalog = "workspace"
schema  = "medallion_cdc_orq"
volume  = "raw_data"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"
landing   = f"{base_path}/landing"     # CDC feed events are landed here

# Ensure the required structure exists (idempotent)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}")
spark.sql(f"USE {catalog}.{schema}")

print("catalog/schema :", f"{catalog}.{schema}")
print("landing        :", landing)

catalog/schema : workspace.medallion_cdc_orq
landing        : /Volumes/workspace/medallion_cdc_orq/raw_data/landing


## Bronze Layer (Landing the Feed As-Is)

The Bronze layer is a **faithful copy** of the CDC feed. We store *every change event* exactly as it arrives,
including the `operation` and `sequence_num` columns. **Nothing is collapsed or transformed.**

We rebuild the Bronze table by reading **the entire landing directory** on every run, ensuring that the complete
history of the feed remains available and reproducible within the Job.

In [0]:
# BRONZE: Ingest ALL events from the landing directory without transformations
from pyspark.sql.functions import current_timestamp, col

raw_df = (
    spark.read
      .option("multiLine", "false")     # JSON Lines: one JSON object per line
      .json(landing)
)

bronze_df = (
    raw_df
      .withColumn("_ingested_at", current_timestamp())
      .withColumn("_source_file", col("_metadata.file_path"))
)

# Rebuild Bronze from the entire available feed (idempotent within the Job)
(bronze_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_orders_cdc"))

print("Bronze rebuilt from the entire landing directory.")
spark.sql("SELECT order_id, status, operation, sequence_num FROM bronze_orders_cdc ORDER BY sequence_num").show()

Bronze rebuilt from the entire landing directory.
+--------+-------+---------+------------+
|order_id| status|operation|sequence_num|
+--------+-------+---------+------------+
|       1|pending|   INSERT|           1|
|       2|pending|   INSERT|           2|
|       3|pending|   INSERT|           3|
+--------+-------+---------+------------+



## Silver Layer (Applying the Changes) — "The Real CDC"

The Silver layer represents the **current state** of the data: **one row per `order_id`** containing its latest version.

The CDC pattern consists of two steps:

1. **Deduplicate the feed** by keeping **only the latest event for each `order_id`** based on
   `sequence_num` (a `MERGE` statement cannot process multiple rows with the same key simultaneously).
2. **`MERGE INTO`**: perform an upsert for all non-`DELETE` operations and remove rows when the operation is `DELETE`.

In [0]:
%sql
-- Silver table: the current state (without CDC feed metadata)
CREATE TABLE IF NOT EXISTS silver_orders (
  order_id  INT,
  customer  STRING,
  product   STRING,
  amount    DOUBLE,
  status    STRING
)

In [0]:
# SILVER: Apply the CDC feed using window-based deduplication + MERGE INTO
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

bronze = spark.table("bronze_orders_cdc")

# 1) Keep the latest event per key (order_id) based on the sequence number
w = Window.partitionBy("order_id").orderBy(col("sequence_num").desc())
latest = (
    bronze
      .withColumn("_rn", row_number().over(w))
      .filter(col("_rn") == 1)          # Keep only the most recent change per order
      .drop("_rn")
)
latest.createOrReplaceTempView("latest_cdc")

# 2) MERGE: Apply the latest state to Silver
spark.sql("""
    MERGE INTO silver_orders AS t
    USING latest_cdc         AS s
      ON t.order_id = s.order_id
    WHEN MATCHED AND s.operation = 'DELETE' THEN DELETE
    WHEN MATCHED AND s.operation != 'DELETE' THEN UPDATE SET
        t.customer = s.customer, t.product = s.product,
        t.amount   = s.amount,   t.status  = s.status
    WHEN NOT MATCHED AND s.operation != 'DELETE' THEN INSERT
        (order_id, customer, product, amount, status)
        VALUES (s.order_id, s.customer, s.product, s.amount, s.status)
""")

print("CDC applied to silver_orders.")

CDC applied to silver_orders.


In [0]:
%sql
-- Silver: the CURRENT STATE after applying the CDC changes
SELECT order_id, customer, product, amount, status
FROM silver_orders
ORDER BY order_id

order_id,customer,product,amount,status
1,Ana,Sneakers,90.0,pending
2,Beto,T-Shirt,25.0,pending
3,Carla,Cap,15.0,pending


## Gold Layer (Business Aggregation)

The Gold layer no longer operates on individual rows. Instead, it aggregates the current state stored in the Silver layer into **business metrics**. Since it is built on top of the already consolidated Silver table, any order removed by a `DELETE` operation **is not included** in the results—which is exactly the desired behavior.

In [0]:
%sql
-- GOLD: Business metrics based on the current state
CREATE OR REPLACE TABLE gold_order_summary AS
SELECT
  status,
  COUNT(*)              AS num_orders,
  ROUND(SUM(amount), 2) AS total_amount
FROM silver_orders
GROUP BY status
ORDER BY status;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM gold_order_summary
ORDER BY status;

status,num_orders,total_amount
pending,3,130.0


## The Production Version: `AUTO CDC INTO` (Reference Only)

Everything we implemented manually using `MERGE INTO` and window-based deduplication is **exactly what Databricks automates** with `AUTO CDC INTO` (formerly `APPLY CHANGES INTO`). However, this functionality must run **inside a Lakeflow Declarative Pipelines pipeline**, not within a notebook task.

```sql
-- (This belongs INSIDE a Lakeflow Declarative Pipelines pipeline)
CREATE OR REFRESH STREAMING TABLE silver_orders;

CREATE FLOW silver_orders_flow AS AUTO CDC INTO silver_orders
FROM STREAM(bronze_orders_cdc)
KEYS (order_id)
APPLY AS DELETE WHEN operation = "DELETE"
SEQUENCE BY sequence_num
COLUMNS * EXCEPT (operation, sequence_num);
```

| Our Manual Implementation | `AUTO CDC INTO` Clause |
|---|---|
| `MERGE ... ON t.order_id = s.order_id` | `KEYS (order_id)` |
| `WHEN MATCHED AND operation='DELETE' THEN DELETE` | `APPLY AS DELETE WHEN operation="DELETE"` |
| `Window.partitionBy(...).orderBy(sequence_num.desc())` + `row_number` | `SEQUENCE BY sequence_num` |
| Upsert (`INSERT` / `UPDATE SET`) | Default behavior |
| `.drop("operation", "sequence_num")` | `COLUMNS * EXCEPT (operation, sequence_num)` |

---

**Task 2 completed.** The Bronze, Silver, and Gold layers have been updated.

**Task 3** (`Verify_Output`) will read the Gold layer to confirm the final results.